# Data Quality and Validation

Data quality is crucial for reliable data engineering pipelines. This notebook covers data validation, quality checks, and data profiling techniques.

## What is Data Quality?

Data quality encompasses:
- **Accuracy**: Data correctly represents reality
- **Completeness**: No missing required data
- **Consistency**: Data is consistent across systems
- **Timeliness**: Data is up-to-date
- **Validity**: Data conforms to defined formats
- **Uniqueness**: No duplicate records

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re

# Sample dataset with quality issues
data = {
    'customer_id': [1, 2, 3, 4, 5, 5, 7, 8, None, 10],  # Duplicate and null
    'name': ['Alice', 'Bob', 'Charlie', None, 'Eve', 'Eve', 'Frank', 'Grace', 'Henry', 'Ivy'],
    'email': ['alice@example.com', 'bob@test', 'charlie@example.com', 
              'diana@example.com', 'eve@example.com', 'eve@example.com',
              'frank@example.com', 'GRACE@EXAMPLE.COM', 'henry@example.com', 'ivy@example.com'],
    'age': [25, 30, -5, 35, 28, 28, 150, 42, 33, 29],  # Invalid ages
    'country': ['USA', 'uk', 'CANADA', 'USA', 'UK', 'UK', 'usa', 'Canada', 'USA', None],  # Inconsistent
    'signup_date': ['2025-01-01', '2025-01-15', '2025-02-30', '2025-01-20',  # Invalid date
                   '2025-01-25', '2025-01-25', '2025-02-01', '2025-02-05',
                   '2025-02-10', '2025-02-15'],
    'revenue': [1000, 2500, 3000, None, 1500, 1500, 4000, 3500, 2000, 2200]
}

df = pd.DataFrame(data)
print("Sample Dataset:")
print(df)

## Data Quality Checks

### 1. Completeness Checks

In [ ]:
class CompletenessChecker:
    """Check data completeness"""
    
    @staticmethod
    def check_nulls(df, column=None):
        """Check for null values"""
        if column:
            null_count = df[column].isnull().sum()
            null_pct = (null_count / len(df)) * 100
            return {
                'column': column,
                'null_count': null_count,
                'null_percentage': f"{null_pct:.2f}%"
            }
        else:
            nulls = df.isnull().sum()
            result = []
            for col in df.columns:
                if nulls[col] > 0:
                    pct = (nulls[col] / len(df)) * 100
                    result.append({
                        'column': col,
                        'null_count': nulls[col],
                        'null_percentage': f"{pct:.2f}%"
                    })
            return result
    
    @staticmethod
    def check_required_fields(df, required_fields):
        """Verify required fields are present and not null"""
        issues = []
        for field in required_fields:
            if field not in df.columns:
                issues.append(f"Missing column: {field}")
            elif df[field].isnull().any():
                null_count = df[field].isnull().sum()
                issues.append(f"{field}: {null_count} null values")
        return issues

checker = CompletenessChecker()

print("\nNull Value Analysis:")
nulls = checker.check_nulls(df)
for item in nulls:
    print(f"  {item['column']}: {item['null_count']} ({item['null_percentage']})")

print("\nRequired Fields Check:")
required = ['customer_id', 'name', 'email']
issues = checker.check_required_fields(df, required)
if issues:
    for issue in issues:
        print(f"  ✗ {issue}")
else:
    print("  ✓ All required fields present")

### 2. Uniqueness Checks

In [ ]:
class UniquenessChecker:
    """Check data uniqueness"""
    
    @staticmethod
    def check_duplicates(df, columns=None):
        """Check for duplicate records"""
        if columns:
            duplicates = df[df.duplicated(subset=columns, keep=False)]
        else:
            duplicates = df[df.duplicated(keep=False)]
        
        return {
            'duplicate_count': len(duplicates),
            'duplicate_percentage': f"{(len(duplicates)/len(df))*100:.2f}%",
            'duplicates': duplicates
        }
    
    @staticmethod
    def check_unique_constraint(df, column):
        """Verify column has unique values"""
        unique_count = df[column].nunique()
        total_count = len(df)
        is_unique = unique_count == total_count
        
        return {
            'column': column,
            'is_unique': is_unique,
            'unique_values': unique_count,
            'total_values': total_count
        }

uniqueness = UniquenessChecker()

print("\nDuplicate Records:")
dup_result = uniqueness.check_duplicates(df, ['customer_id', 'email'])
print(f"  Found: {dup_result['duplicate_count']} ({dup_result['duplicate_percentage']})")
if dup_result['duplicate_count'] > 0:
    print("\n  Duplicate records:")
    print(dup_result['duplicates'][['customer_id', 'name', 'email']])

print("\nUnique Constraint Check:")
unique_check = uniqueness.check_unique_constraint(df, 'customer_id')
if unique_check['is_unique']:
    print(f"  ✓ customer_id is unique")
else:
    print(f"  ✗ customer_id has duplicates")
    print(f"    Unique: {unique_check['unique_values']}, Total: {unique_check['total_values']}")

### 3. Validity Checks

In [ ]:
class ValidityChecker:
    """Check data validity"""
    
    @staticmethod
    def check_email_format(df, column):
        """Validate email format"""
        email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
        invalid = df[~df[column].str.match(email_pattern, na=False)]
        
        return {
            'invalid_count': len(invalid),
            'invalid_records': invalid
        }
    
    @staticmethod
    def check_range(df, column, min_val, max_val):
        """Check if values are within valid range"""
        out_of_range = df[(df[column] < min_val) | (df[column] > max_val)]
        
        return {
            'out_of_range_count': len(out_of_range),
            'out_of_range_records': out_of_range
        }
    
    @staticmethod
    def check_date_validity(df, column):
        """Check for invalid dates"""
        invalid_dates = []
        for idx, val in df[column].items():
            try:
                pd.to_datetime(val)
            except:
                invalid_dates.append((idx, val))
        
        return {
            'invalid_count': len(invalid_dates),
            'invalid_dates': invalid_dates
        }
    
    @staticmethod
    def check_allowed_values(df, column, allowed_values):
        """Check if values are from allowed set"""
        invalid = df[~df[column].isin(allowed_values) & df[column].notna()]
        
        return {
            'invalid_count': len(invalid),
            'invalid_values': invalid[column].unique().tolist()
        }

validator = ValidityChecker()

print("\nEmail Format Validation:")
email_check = validator.check_email_format(df, 'email')
print(f"  Invalid emails: {email_check['invalid_count']}")
if email_check['invalid_count'] > 0:
    print("  Invalid records:")
    print(email_check['invalid_records'][['customer_id', 'email']])

print("\nAge Range Validation (0-120):")
age_check = validator.check_range(df, 'age', 0, 120)
print(f"  Out of range: {age_check['out_of_range_count']}")
if age_check['out_of_range_count'] > 0:
    print("  Invalid ages:")
    print(age_check['out_of_range_records'][['customer_id', 'name', 'age']])

print("\nDate Validity:")
date_check = validator.check_date_validity(df, 'signup_date')
print(f"  Invalid dates: {date_check['invalid_count']}")
if date_check['invalid_count'] > 0:
    for idx, val in date_check['invalid_dates']:
        print(f"    Row {idx}: {val}")

### 4. Consistency Checks

In [ ]:
class ConsistencyChecker:
    """Check data consistency"""
    
    @staticmethod
    def check_standardization(df, column):
        """Check for inconsistent formatting"""
        unique_values = df[column].dropna().unique()
        standardized = set(str(v).upper() for v in unique_values)
        
        needs_standardization = len(unique_values) > len(standardized)
        
        return {
            'needs_standardization': needs_standardization,
            'unique_values': len(unique_values),
            'standardized_values': len(standardized),
            'values': sorted(unique_values)
        }
    
    @staticmethod
    def standardize_column(df, column, standardization_func):
        """Standardize column values"""
        df[column] = df[column].apply(standardization_func)
        return df

consistency = ConsistencyChecker()

print("\nCountry Standardization Check:")
country_check = consistency.check_standardization(df, 'country')
print(f"  Unique values: {country_check['unique_values']}")
print(f"  After standardization: {country_check['standardized_values']}")
print(f"  Values: {country_check['values']}")
if country_check['needs_standardization']:
    print("  ⚠ Standardization needed")

## Data Cleaning and Fixing

Apply fixes to data quality issues:

In [ ]:
class DataCleaner:
    """Clean and fix data quality issues"""
    
    @staticmethod
    def remove_duplicates(df, columns=None):
        """Remove duplicate records"""
        before = len(df)
        if columns:
            df_clean = df.drop_duplicates(subset=columns)
        else:
            df_clean = df.drop_duplicates()
        after = len(df_clean)
        print(f"Removed {before - after} duplicate records")
        return df_clean
    
    @staticmethod
    def fix_nulls(df, column, strategy='drop', fill_value=None):
        """Fix null values"""
        if strategy == 'drop':
            return df.dropna(subset=[column])
        elif strategy == 'fill':
            df[column] = df[column].fillna(fill_value)
            return df
        elif strategy == 'forward_fill':
            df[column] = df[column].ffill()
            return df
    
    @staticmethod
    def standardize_text(df, column, to_case='upper'):
        """Standardize text values"""
        if to_case == 'upper':
            df[column] = df[column].str.upper()
        elif to_case == 'lower':
            df[column] = df[column].str.lower()
        elif to_case == 'title':
            df[column] = df[column].str.title()
        return df
    
    @staticmethod
    def fix_out_of_range(df, column, min_val, max_val, strategy='clip'):
        """Fix out of range values"""
        if strategy == 'clip':
            df[column] = df[column].clip(min_val, max_val)
        elif strategy == 'null':
            df.loc[(df[column] < min_val) | (df[column] > max_val), column] = None
        return df

# Create a clean copy
df_clean = df.copy()

cleaner = DataCleaner()

print("\nCleaning Data...\n")

# Remove duplicates
df_clean = cleaner.remove_duplicates(df_clean, columns=['customer_id'])

# Standardize country
df_clean = cleaner.standardize_text(df_clean, 'country', to_case='upper')
print("Standardized country names")

# Standardize email
df_clean = cleaner.standardize_text(df_clean, 'email', to_case='lower')
print("Standardized email addresses")

# Fix age values
df_clean = cleaner.fix_out_of_range(df_clean, 'age', 0, 120, strategy='null')
print("Fixed out-of-range ages")

print("\nCleaned Dataset:")
print(df_clean)

## Data Quality Report

Generate comprehensive quality report:

In [ ]:
class DataQualityReport:
    """Generate comprehensive data quality report"""
    
    @staticmethod
    def generate_report(df, name="Dataset"):
        """Generate full quality report"""
        print(f"\n{'=' * 70}")
        print(f"DATA QUALITY REPORT: {name}")
        print(f"{'=' * 70}\n")
        
        # Basic statistics
        print("BASIC STATISTICS")
        print(f"  Total Records: {len(df)}")
        print(f"  Total Columns: {len(df.columns)}")
        print(f"  Memory Usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
        
        # Completeness
        print("\nCOMPLETENESS")
        total_cells = len(df) * len(df.columns)
        null_cells = df.isnull().sum().sum()
        completeness = ((total_cells - null_cells) / total_cells) * 100
        print(f"  Overall Completeness: {completeness:.2f}%")
        
        null_cols = df.isnull().sum()
        null_cols = null_cols[null_cols > 0]
        if len(null_cols) > 0:
            print("  Columns with nulls:")
            for col, count in null_cols.items():
                pct = (count / len(df)) * 100
                print(f"    {col}: {count} ({pct:.1f}%)")
        
        # Uniqueness
        print("\nUNIQUENESS")
        for col in df.columns:
            unique_count = df[col].nunique()
            unique_pct = (unique_count / len(df)) * 100
            print(f"  {col}: {unique_count} unique ({unique_pct:.1f}%)")
        
        # Data types
        print("\nDATA TYPES")
        for col in df.columns:
            print(f"  {col}: {df[col].dtype}")
        
        # Quality score
        print("\nQUALITY SCORE")
        score = completeness  # Simple score based on completeness
        
        if score >= 95:
            grade = "A (Excellent)"
        elif score >= 85:
            grade = "B (Good)"
        elif score >= 75:
            grade = "C (Fair)"
        else:
            grade = "D (Poor)"
        
        print(f"  Score: {score:.2f}%")
        print(f"  Grade: {grade}")
        
        print(f"\n{'=' * 70}\n")

# Generate reports
report = DataQualityReport()
report.generate_report(df, "Original Dataset")
report.generate_report(df_clean, "Cleaned Dataset")

## Data Profiling

Profile data to understand its characteristics:

In [ ]:
class DataProfiler:
    """Profile dataset characteristics"""
    
    @staticmethod
    def profile_numeric(df, column):
        """Profile numeric column"""
        return {
            'mean': df[column].mean(),
            'median': df[column].median(),
            'std': df[column].std(),
            'min': df[column].min(),
            'max': df[column].max(),
            'q25': df[column].quantile(0.25),
            'q75': df[column].quantile(0.75)
        }
    
    @staticmethod
    def profile_categorical(df, column):
        """Profile categorical column"""
        value_counts = df[column].value_counts()
        return {
            'unique_values': df[column].nunique(),
            'most_common': value_counts.index[0] if len(value_counts) > 0 else None,
            'most_common_count': value_counts.iloc[0] if len(value_counts) > 0 else 0,
            'distribution': value_counts.to_dict()
        }

profiler = DataProfiler()

print("\nNumeric Profile - Age:")
age_profile = profiler.profile_numeric(df_clean, 'age')
for key, value in age_profile.items():
    if value is not None:
        print(f"  {key}: {value:.2f}")

print("\nCategorical Profile - Country:")
country_profile = profiler.profile_categorical(df_clean, 'country')
print(f"  Unique values: {country_profile['unique_values']}")
print(f"  Most common: {country_profile['most_common']} ({country_profile['most_common_count']} times)")
print("  Distribution:")
for value, count in country_profile['distribution'].items():
    print(f"    {value}: {count}")

## Data Quality Framework

Complete framework for data quality:

In [ ]:
class DataQualityFramework:
    """Comprehensive data quality framework"""
    
    def __init__(self):
        self.checks = []
        self.results = []
    
    def add_check(self, name, check_func, *args, **kwargs):
        """Add a quality check"""
        self.checks.append({
            'name': name,
            'function': check_func,
            'args': args,
            'kwargs': kwargs
        })
    
    def run_checks(self, df):
        """Run all quality checks"""
        print(f"\nRunning {len(self.checks)} data quality checks...\n")
        
        for check in self.checks:
            try:
                result = check['function'](df, *check['args'], **check['kwargs'])
                status = 'PASS' if self._evaluate_result(result) else 'FAIL'
                self.results.append({
                    'check': check['name'],
                    'status': status,
                    'result': result
                })
                print(f"  [{status}] {check['name']}")
            except Exception as e:
                self.results.append({
                    'check': check['name'],
                    'status': 'ERROR',
                    'result': str(e)
                })
                print(f"  [ERROR] {check['name']}: {str(e)}")
    
    def _evaluate_result(self, result):
        """Evaluate check result"""
        if isinstance(result, dict):
            if 'invalid_count' in result:
                return result['invalid_count'] == 0
            if 'is_unique' in result:
                return result['is_unique']
        return True
    
    def get_summary(self):
        """Get summary of check results"""
        total = len(self.results)
        passed = sum(1 for r in self.results if r['status'] == 'PASS')
        failed = sum(1 for r in self.results if r['status'] == 'FAIL')
        errors = sum(1 for r in self.results if r['status'] == 'ERROR')
        
        return {
            'total': total,
            'passed': passed,
            'failed': failed,
            'errors': errors,
            'pass_rate': f"{(passed/total)*100:.1f}%" if total > 0 else "0%"
        }

# Example usage
framework = DataQualityFramework()

# Add checks
framework.add_check("Email format", validator.check_email_format, 'email')
framework.add_check("Age range", validator.check_range, 'age', 0, 120)
framework.add_check("Customer ID unique", uniqueness.check_unique_constraint, 'customer_id')

# Run checks
framework.run_checks(df_clean)

# Print summary
summary = framework.get_summary()
print(f"\nSummary:")
print(f"  Total checks: {summary['total']}")
print(f"  Passed: {summary['passed']}")
print(f"  Failed: {summary['failed']}")
print(f"  Errors: {summary['errors']}")
print(f"  Pass rate: {summary['pass_rate']}")

## Best Practices

1. **Define Quality Rules**: Clearly define what "quality" means for your data
2. **Automate Checks**: Integrate quality checks into your pipelines
3. **Monitor Continuously**: Track quality metrics over time
4. **Document Issues**: Log and track data quality issues
5. **Set Thresholds**: Define acceptable quality thresholds
6. **Alert on Failures**: Notify stakeholders of quality issues
7. **Root Cause Analysis**: Investigate and fix underlying issues
8. **Data Lineage**: Track data from source to destination

## Exercises

1. Create custom validation rules for your domain
2. Implement a data quality dashboard
3. Build a quality scoring system
4. Create automated data quality reports
5. Implement data quality SLAs (Service Level Agreements)